# Popularity groups, user profiles, segments, and history buckets

This notebook completes the data-preparation step (Step 1) of the methodology and produces the input tables that the UPD evaluator in `upd-evaluation.ipynb` expects. It uses the **training split only** for every quantity below, so no test information leaks into popularity labels or user profiles.

It rebuilds the same per-user 80/20 split as `als-baseline.ipynb` (same hash-based ordering, seed 42). A count check confirms that the rebuilt split matches the one used for the ALS baseline. It does not retrain ALS and does not need the baseline notebook's live session.

**Outputs** (saved under `outputs/popularity-profiles/<run timestamp>/`):

| Table | Columns |
|---|---|
| `movie-popularity-groups` | `movieId`, `ratingCount`, `cumulativeShare`, `popularityGroup` (H/M/T) |
| `user-popularity-profiles` | `userId`, `head`, `mid`, `tail` |
| `user-segments` | `userId`, `segment` |
| `user-history-buckets` | `userId`, `trainingRatingCount`, `historyBucket` |
| `evaluation-users` | `userId` |

## 1. Environment

Only `spark-sql` is needed here; ALS (`spark-mllib`) is not used. The assertions match the runtime verified in the other notebooks.

In [ ]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`

import org.apache.spark.sql.SparkSession

In [ ]:
val spark = SparkSession.builder()
  .appName("CS 7265 - Popularity Profiles")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
val scalaVersion = scala.util.Properties.versionNumberString
val javaVersion = System.getProperty("java.version")
val checkCount = spark.range(3).count()

assert(spark.version == "3.5.9")
assert(scalaVersion == "2.12.18")
assert(System.getProperty("java.specification.version") == "17")
assert(checkCount == 3L)

println(s"Spark: ${spark.version}")
println(s"Scala: $scalaVersion")
println(s"Java: $javaVersion")
println(s"Environment check passed: $checkCount synthetic rows.")

## 2. Load ratings and rebuild the training/test split

The next cells repeat the loading and splitting code from `als-baseline.ipynb` unchanged. The split is deterministic (the shuffle key is `xxhash64(splitSeed, userId, movieId)`), so it reproduces the same training and test sets on any machine.

In [ ]:
case class Rating(userId: Int, movieId: Int, rating: Float, timestamp: Long)

class RatingParser extends (String => Rating) with Serializable {
  def apply(line: String): Rating = {
    val fields = line.split("::")
    require(fields.length == 4, "Expected four fields per MovieLens rating")

    Rating(
      fields(0).toInt,
      fields(1).toInt,
      fields(2).toFloat,
      fields(3).toLong
    )
  }
}

In [ ]:
val ratingsPath = Seq(
  "data/ml-1m/ratings.dat",
  "../data/ml-1m/ratings.dat"
).find(path => new java.io.File(path).isFile).getOrElse {
  throw new IllegalArgumentException("Place MovieLens 1M ratings.dat in data/ml-1m/ and run from the repository root or notebooks folder.")
}

val sc = spark.sparkContext
val lines = sc.textFile(ratingsPath)
val ratingsRDD = lines.map(new RatingParser)

import spark.implicits._
val ratingsDF = ratingsRDD.toDF()

In [ ]:
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

val trainingFraction = 0.8
val splitSeed = 42L

val perUser = Window.partitionBy("userId")
val ratingsWithCounts = ratingsDF.withColumn(
  "userRatingCount",
  count("*").over(perUser)
)

val ratingsWithKeys = ratingsWithCounts.withColumn(
  "shuffleKey",
  xxhash64(lit(splitSeed), col("userId"), col("movieId"))
)

val shuffledPerUser = Window
  .partitionBy("userId")
  .orderBy(col("shuffleKey"), col("movieId"))

val rankedRatings = ratingsWithKeys.withColumn(
  "ratingPosition",
  row_number().over(shuffledPerUser)
)

val ratingsWithCutoff = rankedRatings.withColumn(
  "trainingCount",
  floor(col("userRatingCount") * trainingFraction)
)

val training = ratingsWithCutoff.filter(
  col("ratingPosition") <= col("trainingCount")
)
val test = ratingsWithCutoff.filter(
  col("ratingPosition") > col("trainingCount")
)

val trainingRatings = training.select("userId", "movieId", "rating", "timestamp")
val testRatings = test.select("userId", "movieId", "rating", "timestamp")

### Check that this is the same split as the ALS baseline

The ALS notebook's saved output reports 797,758 training ratings, 202,451 test ratings, and 6,040 users in each set. Matching all three confirms that the popularity labels and profiles below are computed from exactly the data the baseline model was trained on. The overlap check confirms that no user–movie pair appears in both sets.

In [ ]:
val trainingRatingCount = trainingRatings.count()
val testRatingCount = testRatings.count()
val trainingUserCount = trainingRatings.select("userId").distinct().count()
val testUserCount = testRatings.select("userId").distinct().count()
val overlapCount = trainingRatings.select("userId", "movieId")
  .intersect(testRatings.select("userId", "movieId")).count()

println(s"Training: $trainingRatingCount ratings from $trainingUserCount users")
println(s"Test: $testRatingCount ratings from $testUserCount users")
println(s"Overlapping user-movie pairs: $overlapCount")

assert(trainingRatingCount == 797758L, "Training count differs from the ALS baseline split.")
assert(testRatingCount == 202451L, "Test count differs from the ALS baseline split.")
assert(trainingUserCount == 6040L && testUserCount == 6040L, "Every user must appear in both sets.")
assert(overlapCount == 0L, "Training and test must not overlap.")
println("Split matches the ALS baseline.")

## 3. Label movies Head, Mid, or Tail

Popularity is measured on the training ratings only. Movies are sorted by rating count (most-rated first, ties broken by ascending movie ID so the order is deterministic), and the cumulative share of all training ratings is accumulated down the list.

- **Head (H):** the most-rated movies that together account for the first 20% of ratings. The movie whose rating count carries the cumulative share past 20% is included in Head.
- **Tail (T):** the least-rated movies that together account for the last 20% of ratings, using the same boundary convention from the bottom.
- **Mid (M):** every other movie.

The 20/60/20 split is therefore by share of **ratings**, not share of movies, so Head contains few movies and Tail contains many. Only movies that appear in training receive a label; this covers every movie the ALS model can recommend.

In [ ]:
val headShare = 0.20
val tailShare = 0.20

val totalTrainingRatings = trainingRatings.count().toDouble
val movieCounts = trainingRatings.groupBy("movieId").agg(count("*").as("ratingCount"))

val popularityOrder = Window
  .orderBy(col("ratingCount").desc, col("movieId").asc)
  .rowsBetween(Window.unboundedPreceding, Window.currentRow)

val movieGroups = movieCounts
  .withColumn("cumulativeShare",
    sum(col("ratingCount")).over(popularityOrder) / lit(totalTrainingRatings))
  .withColumn("shareBefore",
    col("cumulativeShare") - col("ratingCount") / lit(totalTrainingRatings))
  .withColumn("popularityGroup",
    when(col("shareBefore") < headShare, "H")
      .when(col("cumulativeShare") > 1.0 - tailShare, "T")
      .otherwise("M"))
  .select("movieId", "ratingCount", "cumulativeShare", "popularityGroup")
  .cache()

val groupSummary = movieGroups.groupBy("popularityGroup").agg(
  count("*").as("movieCount"),
  sum("ratingCount").as("ratingSum")
).withColumn("ratingShare", col("ratingSum") / lit(totalTrainingRatings))
 .orderBy("popularityGroup")

groupSummary.show(truncate = false)

### Check the labels

Every training movie must have exactly one label, and the three groups must cover every training rating exactly once. Head and Tail each hold at least 20% of ratings, and slightly more because the boundary movie is included in its group. The upper bound of 21% catches mistakes without being sensitive to the exact boundary movie.

In [ ]:
val labeledMovieCount = movieGroups.count()
val trainingMovieCount = trainingRatings.select("movieId").distinct().count()
val groupRows = groupSummary.collect().map(r =>
  r.getString(0) -> (r.getLong(1), r.getLong(2), r.getDouble(3))).toMap

assert(labeledMovieCount == trainingMovieCount, "Every training movie needs exactly one label.")
assert(movieGroups.select("movieId").distinct().count() == labeledMovieCount, "Duplicate movie labels.")
assert(groupRows.keySet == Set("H", "M", "T"), "All three groups must be present.")
assert(groupRows.values.map(_._2).sum == trainingRatingCount, "Groups must cover every training rating once.")
assert(groupRows("H")._3 >= headShare - 1e-9 && groupRows("H")._3 < 0.21, "Head share out of range.")
assert(groupRows("T")._3 >= tailShare - 1e-9 && groupRows("T")._3 < 0.21, "Tail share out of range.")

println(s"Labeled $labeledMovieCount movies (H/M/T): " +
  s"${groupRows("H")._1} / ${groupRows("M")._1} / ${groupRows("T")._1}")
println("Movie popularity label checks passed.")

## 4. Build user popularity profiles

Each user's profile `p(u)` is the share of their training ratings that falls in Head, Mid, and Tail movies, weighted by the rating value as in the reference paper. A user who rated one Head movie 5 and one Tail movie 3 has weight 5 on Head and 3 on Tail, giving `p(u) = (0.625, 0, 0.375)`. The three shares sum to 1 by construction.

`trainingRatingCount` (the number of training ratings per user, `n_u`) is kept for the history buckets and for the later history-aware extension.

In [ ]:
val labeledTraining = trainingRatings
  .join(movieGroups.select("movieId", "popularityGroup"), Seq("movieId"), "inner")

assert(labeledTraining.count() == trainingRatingCount, "Joining labels must not drop or duplicate ratings.")

val ratingValue = col("rating").cast("double")
def groupWeight(label: String) =
  sum(when(col("popularityGroup") === label, ratingValue).otherwise(0.0))

val userStats = labeledTraining.groupBy("userId").agg(
  sum(ratingValue).as("ratingTotal"),
  groupWeight("H").as("headWeight"),
  groupWeight("M").as("midWeight"),
  groupWeight("T").as("tailWeight"),
  count("*").as("trainingRatingCount")
)

val userProfiles = userStats.select(
  col("userId"),
  (col("headWeight") / col("ratingTotal")).as("head"),
  (col("midWeight") / col("ratingTotal")).as("mid"),
  (col("tailWeight") / col("ratingTotal")).as("tail"),
  col("trainingRatingCount")
).cache()

val badProfiles = userProfiles.filter(
  abs(col("head") + col("mid") + col("tail") - 1.0) > 1e-9 ||
  col("head") < 0.0 || col("mid") < 0.0 || col("tail") < 0.0
).count()

assert(userProfiles.count() == 6040L, "Every user needs a profile.")
assert(badProfiles == 0L, "Profiles must be nonnegative and sum to 1.")

userProfiles.agg(
  avg("head").as("meanHead"), avg("mid").as("meanMid"), avg("tail").as("meanTail"),
  min("trainingRatingCount").as("minTrainingRatings"),
  max("trainingRatingCount").as("maxTrainingRatings")
).show(truncate = false)
println("User profile checks passed.")

## 5. Assign user segments and history buckets

**Segments.** Users are ranked by Head share (highest first, ties broken by user ID) and cut into three equal-sized groups with `ntile(3)`: the top third is *blockbuster-focused*, the middle third *diverse*, and the bottom third *niche-focused*. With 6,040 users the sizes are 2,014, 2,013, and 2,013.

**History buckets.** Based on each user's number of training ratings: *short* (fewer than 50), *medium* (50 to 200 inclusive), and *long* (more than 200).

In [ ]:
val segmentOrder = Window.orderBy(col("head").desc, col("userId").asc)

val userSegments = userProfiles
  .withColumn("tile", ntile(3).over(segmentOrder))
  .withColumn("segment",
    when(col("tile") === 1, "blockbuster-focused")
      .when(col("tile") === 2, "diverse")
      .otherwise("niche-focused"))
  .select("userId", "segment", "head")
  .cache()

val historyBuckets = userProfiles.select(
  col("userId"),
  col("trainingRatingCount"),
  when(col("trainingRatingCount") < 50, "short")
    .when(col("trainingRatingCount") <= 200, "medium")
    .otherwise("long").as("historyBucket")
).cache()

println("Segments (by Head share):")
userSegments.groupBy("segment").agg(
  count("*").as("userCount"),
  min("head").as("minHead"), avg("head").as("meanHead"), max("head").as("maxHead")
).orderBy(col("meanHead").desc).show(truncate = false)

println("History buckets:")
historyBuckets.groupBy("historyBucket").agg(
  count("*").as("userCount"),
  min("trainingRatingCount").as("minN"), max("trainingRatingCount").as("maxN")
).orderBy("minN").show(truncate = false)

println("Segment x history bucket (user counts):")
userSegments.join(historyBuckets, Seq("userId"))
  .groupBy("segment").pivot("historyBucket", Seq("short", "medium", "long"))
  .count().orderBy("segment").show(truncate = false)

val segmentSizes = userSegments.groupBy("segment").count().collect().map(_.getLong(1))
assert(userSegments.count() == 6040L && historyBuckets.count() == 6040L, "Every user needs a segment and bucket.")
assert(segmentSizes.length == 3 && segmentSizes.max - segmentSizes.min <= 1L, "Segments must be equal in size.")
println("Segment and bucket checks passed.")

## 6. Export the evaluator inputs

The tables are written under `outputs/popularity-profiles/<run timestamp>/` (ignored by Git), together with a short `split-config.txt` recording the split settings, so the files can be traced to the split that produced them. The read-back check confirms row counts and that the evaluation cohort matches the training users.

In [ ]:
val repositoryRoot = new java.io.File(ratingsPath).getCanonicalFile
  .getParentFile.getParentFile.getParentFile
val profileRunId = java.time.Instant.now().toString.replace(":", "-")
val profileDirectory = new java.io.File(
  repositoryRoot, s"outputs/popularity-profiles/$profileRunId")

def writeTable(df: org.apache.spark.sql.DataFrame, name: String): String = {
  val path = new java.io.File(profileDirectory, name).getPath
  df.coalesce(1).write.mode("errorifexists").parquet(path)
  path
}

val evaluationUsers = trainingRatings.select("userId").distinct()

val exportPaths = Map(
  "movie-popularity-groups" -> writeTable(movieGroups, "movie-popularity-groups"),
  "user-popularity-profiles" -> writeTable(userProfiles.select("userId", "head", "mid", "tail"), "user-popularity-profiles"),
  "user-segments" -> writeTable(userSegments.select("userId", "segment"), "user-segments"),
  "user-history-buckets" -> writeTable(historyBuckets, "user-history-buckets"),
  "evaluation-users" -> writeTable(evaluationUsers, "evaluation-users")
)

val configText = Seq(
  s"splitSeed=$splitSeed",
  s"trainingFraction=$trainingFraction",
  s"trainingRatings=$trainingRatingCount",
  s"testRatings=$testRatingCount",
  s"headShare=$headShare",
  s"tailShare=$tailShare",
  "historyBuckets=short<50;medium=50-200;long>200",
  "segments=ntile(3) by head share descending, ties by userId"
).mkString("\n") + "\n"
java.nio.file.Files.write(
  new java.io.File(profileDirectory, "split-config.txt").toPath,
  configText.getBytes("UTF-8"))

val expectedRows = Map(
  "movie-popularity-groups" -> trainingMovieCount,
  "user-popularity-profiles" -> 6040L,
  "user-segments" -> 6040L,
  "user-history-buckets" -> 6040L,
  "evaluation-users" -> 6040L
)
expectedRows.foreach { case (name, expected) =>
  val actual = spark.read.parquet(exportPaths(name)).count()
  assert(actual == expected, s"$name has $actual rows; expected $expected.")
  println(s"Verified $name: $actual rows")
}

println(s"Export folder: ${profileDirectory.getPath}")

## 7. Stop the Spark session

Run this cell when finished. The exported tables are what `upd-evaluation.ipynb` loads in its "Use real inputs" section.

In [ ]:
movieGroups.unpersist(blocking = true)
userProfiles.unpersist(blocking = true)
userSegments.unpersist(blocking = true)
historyBuckets.unpersist(blocking = true)

spark.stop()
println("Spark session stopped.")